<h2><font color="#004D7F" size=5>Módulo 4: Stacking</font></h2>


<h1><font color="#004D7F" size=6> 1. Voting</font></h1>
<br><br>
<div style="text-align: right">
<font color="#004D7F" size=3>Manuel Castillo-Cara</font><br>
<font color="#004D7F" size=3>Aprendizaje Automático II</font><br>
<font color="#004D7F" size=3>Universidad Nacional de Educación a Distancia</font>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web personal](https://www.manuelcastillo.eu/udemy/)

---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>


* [1. Algoritmo Voting](#section1)
* [2. Especificaciones sobre Voting](#section2)
* [3. Voting para clasificación](#section3)
   * [3.1. Voting hard](#section31)
   * [3.2. Voting soft](#section32)
* [4. Voting para regresión](#section4)
* [Ejercicios](#sectionEj)

---

<a id="section0"></a>
# <font color="#004D7F">0. Contexto</font>

- Para la regresión, Voting implica hacer una predicción que es el promedio de muchos otros modelos de regresión. 
- En la clasificación:
    - Voting (_hard_) implica sumar los votos y predecir la clase con más votos.
    - Voting (_soft_) implica sumar las probabilidades previstas para las etiquetas de clase y predecir la etiqueta de clase con la mayor probabilidad suma. 

En este tutorial, trabajaremos:
- Voting implica sumar las predicciones realizadas por los modelos de clasificación o promediar las predicciones realizadas por los modelos de regresión.
- Cómo funcionan Voting, cuándo utilizarlos y las limitaciones del enfoque.
- Cómo implementar Voting duro y Voting blando para el modelado predictivo de clasificación.


---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section1"></a>
# <font color="#004D7F"> 1. Algoritmo Voting</font>

- Para la regresión, Voting implica hacer una predicción que es el promedio de muchos otros modelos de regresión. 
- En la clasificación:
    - Voting _hard_ implica sumar los votos y predecir la clase con más votos.
    - Voting (_soft_) implica sumar las probabilidades previstas para las etiquetas de clase y predecir la etiqueta de clase con la mayor probabilidad suma. 

- Voting puede considerarse un metamodelo, un modelo de modelos.
- Por tanto, puede usarse con cualquier modelo de Machine learning.
- La forma óptima para Voting es que los modelos contribuyentes fueran independientes. Desafortunadamente, rara vez sucede esto
- Voting _soft_ es para modelos que predicen las probabilidades de pertenencia a una clase.
    - Aunque pueden requerir la calibración de sus puntuaciones similares a las de probabilidad antes de usarse en el conjunto (por ejemplo, SVM, k-NN y CART).
- Voting duro es para modelos que predicen etiquetas de clases.
- Utilizamos Voting si:
    - Da como resultado un mejor rendimiento que cualquier modelo utilizado en el conjunto.
    - Da como resultado una varianza más baja (estabildiad y generalización) que cualquier modelo utilizado en el conjunto.
  
<figure><center>
  <img src="data/voting.png" width="450" height="250" alt="Gráfica">
  <figcaption><blockquote>Voting vs. Bagging. Extraída de <a href="https://medium.com/@chyun55555/ensemble-learning-voting-and-bagging-with-python-40de683b8ff0">Medium</a></blockquote></figcaption>
</center></figure>


---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section2"></a> 
# <font color="#004D7F"> 2. Especificaciones sobre Voting</font>

La biblioteca de aprendizaje automático Python scikit-learn proporciona una implementación de Voting para el aprendizaje automático a través de las clases `VotingRegressor` y `VotingClassifier`. 

- Se proporciona una lista de modelos base a través del argumento `estimators`.
- Esta es una lista de Python donde cada elemento de la lista es una tupla con el nombre del modelo y la instancia del modelo configurado:
```python
    ...
    models = [('lr',LogisticRegression()),('svm',SVC())]
    ensemble = VotingClassifier(estimators=models)
```

Cada modelo de la lista también puede ser un `Pipeline` con cualquier operación previa:
```python
    ...
    models = [('lr',LogisticRegression()),('svm',make_pipeline(StandardScaler(),SVC()))]
    ensemble = VotingClassifier(estimators=models)
```

Especificamos el tipo de voto en clasificación con el argumento `voting` que puede ser `hard` (por defecto) o `soft`:

```python
    ...
    models = [('lr',LogisticRegression()),('svm',SVC())]
    ensemble = VotingClassifier(estimators=models, voting='soft')
```

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section3"></a> 
# <font color="#004D7F"> 3. Voting para Clasificación</font>

Primero, podemos usar la función `make_classification()` para crear un problema de clasificación binaria sintética con 1000 ejemplos y 20 características de entrada.

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, n_redundant=5, random_state=3)
print(X.shape, y.shape)

<a id="section31"></a> 
## <font color="#004D7F"> 3.1. Voting hard</font>

- Ajustamos cinco versiones diferentes del algoritmo k-NN, con 1, 3, 5, 7 y 9 vecinos.
- Creamos una lista de modelos para evaluar, incluida cada versión independiente de las configuraciones del modelo k-NN y Voting _hard_.
- Así veremos si Voting mejora las predicciones individuales
- La función `get_models()` a continuación crea la lista de modelos que podemos evaluar.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`VotingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`KNeighborsClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
</div>

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier

def get_models():
	models = ???
	# indicar el número de vecinos
	???
	???:
		key = 'knn' + str(n)
		models[key] = ???
	# definimos el voto (hard)
	members = ???
	models['hard_voting'] = ???
	return models

Veamos el siguiente código que ya hemos venido haciendo

In [ ]:
# compare hard voting to standalone classifiers
from numpy import mean
from numpy import std
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold
from matplotlib import pyplot

def get_dataset():
	X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, n_redundant=5, random_state=2)
	return X, y

def evaluate_model(model, X, y):
	cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
	scores = cross_val_score(model, X, y, scoring='accuracy', cv=cv, n_jobs=-1)
	return scores

X, y = get_dataset()
models = get_models()
results, names = list(), list()
for name, model in models.items():
	scores = evaluate_model(model, X, y)
	results.append(scores)
	names.append(name)
	print('>%s %.3f (%.3f)' % (name, mean(scores), std(scores)))

In [ ]:
pyplot.boxplot(results, labels=names, showmeans=True)
pyplot.show()

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

<a id="section32"></a> 
## <font color="#004D7F"> 3.2. Voting soft</font>

- Podemos demostrar Voting _soft_ con el algoritmo SVM.
- El algoritmo SVM no predice probabilidades de forma nativa (aunque se puede configurar para predecir puntuaciones similares a las de probabilidad estableciendo el argumento `probability=True` en la clase `SVC`).
- Ajustamos de 1 a 5 (argumento `degree`) el valor del kernel polinomial (`kernel='poly'`).
- La función `get_models()` a continuación crea la lista de modelos que podemos evaluar.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`SVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)
</div>

In [ ]:
from sklearn.svm import SVC

def get_models():
	models = dict()
	???
		key = 'svm' + str(n)
		models[key] = ???
	# definimos el voto
	members = ???
	models['soft_voting'] = ???
	return models

Podemos evaluar e informar el rendimiento del modelo utilizando una validación cruzada repetida de k veces como lo hicimos en la sección anterior. Uniendo esto, el ejemplo completo se enumera a continuación.

In [ ]:
X, y = get_dataset()
models = get_models()
results, names = list(), list()
for name, model in models.items():
	scores = evaluate_model(model, X, y)
	results.append(scores)
	names.append(name)
	print('>%s %.3f (%.3f)' % (name, mean(scores), std(scores)))

In [ ]:
pyplot.boxplot(results, labels=names, showmeans=True)
pyplot.show()

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section4"></a> 
# <font color="#004D7F"> 4. Voting para Regresión</font>

- Usamos Voting para la regresión con un algoritmo CART.
- Ajustamos cinco versiones diferentes del algoritmo CART con la profundidad de 1 a 5 con el argumento `max_depth`. 
- La función `get_models()` a continuación crea la lista de modelos que podemos evaluar.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`VotingRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingRegressor.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`DecisionTreeRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)
</div>

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor

def get_models():
	models = ???
	for n in range(1,6):
		key = 'cart' + str(n)
		models[key] = ???
	# definimos el voto
	members = ???
	models['voting'] = ???
	return models

Podemos evaluar e informar el rendimiento del modelo utilizando una validación cruzada repetida de k veces como lo hicimos en la sección anterior. Los modelos se evalúan utilizando el error absoluto medio (MAE). Uniendo esto, el ejemplo completo se enumera a continuación.

In [ ]:
from sklearn.model_selection import RepeatedKFold
from sklearn.datasets import make_regression

def get_dataset():
	X, y = make_regression(n_samples=1000, n_features=20, n_informative=15, noise=0.1, random_state=1)
	return X, y

def evaluate_model(model, X, y):
	cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
	scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
	return scores

X, y = get_dataset()
models = get_models()
results, names = list(), list()
for name, model in models.items():
	scores = evaluate_model(model, X, y)
	results.append(scores)
	names.append(name)
	print('>%s %.3f (%.3f)' % (name, mean(scores), std(scores)))

In [ ]:
pyplot.boxplot(results, labels=names, showmeans=True)
pyplot.show()

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: La API de scikit-learn invierte el signo del MAE para transformarlo, de minimizar el error a maximizar el error negativo. Esto significa que los errores positivos de gran magnitud se convierten en grandes errores negativos (por ejemplo, 100 se convierte en -100) y un modelo perfecto no tiene ningún error con un valor de 0,0. También significa que podemos ignorar con seguridad el signo de las puntuaciones MAE medias. 

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="sectionEj"></a>
<h3><font color="#004D7F" size=6> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> Ejercicios</font></h3>

Se proponen las siguientes actividades para consolidar el aprendizaje.

# <font color="#004D7F" size=5>Ejercicio 1</font>
__Hiperparámetros__. Explore diferentes configuraciones de hiperparámetros que veas en la librería sobre Voting y comente los resultados. 

# <font color="#004D7F" size=5>Ejercicio 2</font>
__Evaluar algoritmos__. Utilizar diferentes algoritmos a los que hemos usado en esta unidad y evaluar su impacto con Voting usando hiperparáemtros clave.

# <font color="#004D7F" size=5>Ejercicio 3</font>
__Datasets reales__. Busque un dataset original y verdadero (que no sea sintético) y evalúe el uso de los conceptos vistos en esta unidad. Los conjuntos de datos en pueden ser obtenidos del [repositorio de aprendizaje automático de UCI](https://archive.ics.uci.edu/).

# <font color="#004D7F" size=5>Ejercicio 4</font>
__Búsqueda de la mejor configuración__. Como se ha visto existen diferentes hiperparámetros que pueden ajustar nuestro modelo. Haga una búsqueda para un dataset real de cuales, entre un rango amplio de hiperparétros, maximizan la métrica. Puede utilizar una búsqueda aleatoria.

# <font color="#004D7F" size=5>Ejercicio 5</font>
__Algoritmos individuales__. En este ejemplo hemos usado Voting de forma muy sencilla probándolo con un mismo algoritmo. Utliza Voting con una batería de algoritmos diferentes, jugando también con los propios hiperparámentros de cada uno.

# <font color="#004D7F" size=5>Ejercicio 6</font>
__Preprocesamiento de datos__. En este ejemplo hemos usado Voting sin usar ningún procesamiento de datos. Usa el `Pipeline` usando varios algoritmos con el procesamiento de datos apropiado a cada uno. Puede usar varios hiperparámetros propios de cada algoritmo individual que se utilice.

# <font color="#004D7F" size=5>Ejercicio 7</font>
__Incluir más niveles__. En este ejemplo hemos usado el modelo ingresando un solo niel 0 y 1. Se pueden integrar más niveles haciendo una estrategia de integración entre ellos. Se pide integregar más niveles en un dataset real y explicando las limitaciones y fortalezas observadas.

---

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>